##AI-Powered HR Assistant: A Use Case for Nestle’s HR Policy Documents
#Aamir Mohammed

In [1]:
!pip -q install --upgrade --force-reinstall "langchain==0.2.5" "langchain-community==0.2.5" "langchain-openai==0.1.7" "chromadb==0.4.24" "PyPDF2==3.0.1" "pydantic==2.7.4" "numpy==1.26.6"

ERROR: Ignored the following yanked versions: 2.4.0
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement numpy==1.26.6 (from versions: 1.3.0, 1.4.1, 1.5.0, 1.5.1, 1.6.0, 1.6.1, 1.6.2, 1.7.0, 1.7.1, 1.7.2, 1.8.0, 1.8.1, 1.8.2, 1.9.0, 1.9.1, 1.9.2, 1.9.3, 1.10.0.post2, 1.10.1, 1.10.2, 1.10.4, 1.11.0, 1.11.1, 1.11.2, 1.11.3, 1.12.0, 1.12.1, 1.13.0, 1.13.1, 1.13.3, 1.14.0, 1.14.1, 1.14.2, 1.14.3, 1.14.4, 1.14.5, 1.14.6, 1.15.0, 1.15.1, 1.15.2, 1.15.3, 1.15.4, 1.16.0, 1.16.1, 1.16.2, 1.16.3, 1.16.4, 1.16.5, 1.16.6, 1.17.0, 1.17.1, 1.17.2, 1.17.3, 1.17.4, 1.17.5, 1.18.0, 1.18.1, 1.18.2, 1.18.3, 1.18.4, 1.18.5, 1.19.0, 1.19.1, 1.19.2, 1.19.3, 1.19.4, 1.19.5, 1.20.0, 1.20.1, 1.20.2, 1.20.3, 1.21.0, 1.21.1, 1.22.0, 1.22.1, 1.22.2, 

In [2]:
!pip -q install gradio pypdf

In [3]:
import os
import warnings
warnings.filterwarnings("ignore")
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
import time
import gradio as gr

In [4]:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

In [5]:
print("Key loaded:", os.environ.get("OPENAI_API_KEY") is not None)

Key loaded: True


In [6]:
pdf = "/content/the_nestle_hr_policy_pdf_2012.pdf"

loader = PyPDFLoader(pdf)
pages = loader.load()

print("Pages Loaded: ", len(pages))
print(pages[0].page_content[:400])

Pages Loaded:  8
Policy
Mandatory
September   2012
The Nestlé  
Human Resources Policy


In [7]:
#split pdf into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150)

chunks = text_splitter.split_documents(pages)

chunks = [
    i for i in chunks
    if len(i.page_content.strip()) > 200
]

print(f"Number of Chunks: {len(chunks)}")
print(f"First Chunk:\n {chunks[0].page_content[:400]}")
print("\nMetadata: ", chunks[0].metadata)



Number of Chunks: 24
First Chunk:
 Policy
Mandatory
September 
 20
12
Issuing  departement
Hum
an Resources
T arget  audience  
All
 employees
Approver
Executive Board, Nestlé S.A.
Repository
All Nestlé Principles and Policies, Standards and  
Guidelines can be found in the Centre online repository at:  
http://intranet.nestle.com/nestledocs
Copyright
 and  confidentiality
Al
l rights belong to Nestec Ltd., Vevey, Switzerland.
© 20

Metadata:  {'source': '/content/the_nestle_hr_policy_pdf_2012.pdf', 'page': 1}


In [8]:
#create embeddings
openai_embed = OpenAIEmbeddings(model="text-embedding-3-small")

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=openai_embed,
    persist_directory="chroma_nestle_HR"
)

retriever = vectordb.as_retriever(search_kwargs={"k": 6})

print("Retriever Initialized")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Retriever Initialized


In [10]:
from langchain.prompts import PromptTemplate

template = """
You are an HR policy assistant for Nestle.
Answer ONLY using the provided context from the HR policy.
If multiple relevant statements exist, prioritize the most explicit policy language.
If the answer is not in the document, you can say: "I couldn't find that specific HR policy in the Document."
Always include citations as (page X) for the sentences you used.
Context: {context}
Question: {question}
Answer:
"""

In [11]:
# chatgpt for asnwering
llm = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo")

prompt = PromptTemplate(template=template, input_variables=["context", "question"])

def answer_question(question):

    results = retriever.get_relevant_documents(question)

    context = "\n\n".join(
        [f"(page {d.metadata.get('page')}) {d.page_content}" for d in results]
    )

    response = llm.invoke(
        prompt.format(context=context, question=question)
    )

    return response.content



In [14]:
print(answer_question("Does Nestlé tolerate harassment or discrimination?"))

Nestlé does not tolerate any form of harassment or discrimination (page 4).


In [13]:
def HR_answer_controls(question, temperature, max_tokens):
  llm.temperature = float(temperature)
  llm.model_kwargs = {"max_tokens": int(max_tokens)}
  return answer_question(question)

hr_interface = gr.Interface(
    fn=HR_answer_controls,
    inputs=[
        gr.Textbox(lines=2, label="What is your question?"),
        gr.Slider(0, 1, value=0.0, step=0.1, label="Temperature"),
        gr.Slider(50, 250, value=100, step=50, label="Max Tokens"),
    ],
    outputs=gr.Textbox(lines=8, label="Answer"),
    title="Nestlé HR Policy Assistant",
    description="Asks questions based on the uploaded Nestlé HR policy PDF."
)

hr_interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ceec0413cca436dcfa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
